## 1. Imports & Circuit Values

Circuit parameters used to generate the fake data:
- **R** = 150 Ω
- **L** = 15 mH
- **V_max** = 1 V (peak input voltage)
- **Frequency sweep**: 50 Hz – 10 000 Hz

Known capacitances for comparison after back-calculation:
| Label | C (µF) | Expected f₀ (Hz) |
|---|---|---|
| C1uF | 1.0 | 1299.5 |
| C2_2uF | 2.2 | 876.1 |
| C4_7uF | 4.7 | 599.4 |
| C10uF | 10.0 | 410.9 |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Circuit values
R     = 150       # ohms
L     = 15e-3     # henries
V_max = 1.0       # volts (peak input)

# Frequency sweep range
F_MIN, F_MAX = 50, 10_000  # Hz

# Known capacitances for comparison (label -> farads)
KNOWN_C = {
    "C1uF":   1.0e-6,
    "C2_2uF": 2.2e-6,
    "C4_7uF": 4.7e-6,
    "C10uF":  10.0e-6,
}

## 2. Load Data Files

Scans  for all  files.  
In the real project this folder will contain several thousand files — the glob approach handles that automatically.

In [ ]:
DATA_DIR = Path("fake_data")

files = sorted(DATA_DIR.glob("rlc_sweep_*.csv"))
print(f"Found {len(files)} files:")
for f in files:
    print(" ", f.name)

# Load all into a dict keyed by file stem
data = {f.stem: pd.read_csv(f) for f in files}

## 3. Plot All Files

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for name, df in data.items():
    ax.semilogx(df["frequency_hz"], df["gain_db"], label=name)

ax.axhline(-3, color="gray", linestyle="--", linewidth=0.8, label="-3 dB")
ax.set_title("RLC Frequency Sweep — Gain (dB)")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Gain (dB)")
ax.legend()
ax.grid(True, which="both", alpha=0.4)
plt.tight_layout()
plt.show()

## 4. Find Peak Frequency for Each File

The frequency at which gain is maximum is the resonant frequency f₀.

In [ ]:
peaks = {}
for name, df in data.items():
    idx = df["gain"].idxmax()
    peaks[name] = df.loc[idx, "frequency_hz"]

peak_df = pd.DataFrame.from_dict(peaks, orient="index", columns=["peak_freq_hz"])
peak_df.index.name = "file"
peak_df

## 5. Back-Calculate Capacitance

Using the resonant frequency formula:  
**f₀ = 1 / (2π √(LC))**  
→ **C = 1 / (L · (2π f₀)²)**

In [ ]:
rows = []
for name, f0 in peaks.items():
    C_calc = 1 / (L * (2 * np.pi * f0) ** 2)
    label  = name.replace("rlc_sweep_", "")
    C_known = KNOWN_C.get(label, None)
    rows.append({
        "file":          name,
        "peak_freq_hz":  f0,
        "C_calculated_uF": round(C_calc * 1e6, 4),
        "C_known_uF":    round(C_known * 1e6, 4) if C_known else None,
        "error_%":       round(abs(C_calc - C_known) / C_known * 100, 2) if C_known else None,
    })

pd.DataFrame(rows)